#  RAG Pipeline — VS Code Notebook

**Steps:**
1. Set your `OPENAI_API_KEY` in Cell 2
2. Run all cells top to bottom
3. Use the chatbot in the last cell

> Type `chunks off` / `chunks on` at the prompt to toggle chunk display.  
> Type `exit` to stop the chatbot.

In [ ]:
# ============================================================
# 📦 INSTALLS
# ============================================================

%pip install -q openai chromadb pypdf tiktoken

In [ ]:
# ============================================================
# 📚 IMPORTS
# ============================================================

import os
import uuid
import textwrap

from openai import OpenAI
from pypdf import PdfReader

import chromadb

In [ ]:
# ============================================================
# 🔑 OPENAI CLIENT
# Option A: set key directly (not recommended for shared notebooks)
# Option B: set via environment variable before launching VS Code:
#   export OPENAI_API_KEY="sk-..."   (Mac/Linux)
#   $env:OPENAI_API_KEY="sk-..."     (PowerShell)
# ============================================================

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")

# ── Uncomment the line below to set key directly ────────────
# OPENAI_API_KEY = "sk-..."

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY is not set. See instructions above.")

client = OpenAI(api_key=OPENAI_API_KEY)
print("✅ OpenAI client ready.")

In [ ]:
# ============================================================
# 📄 LOAD PDF
# ============================================================

def load_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        extracted = page.extract_text()
        if extracted:
            text += extracted + "\n"
    return text

In [ ]:
# ============================================================
# ✂️ CHUNKER WITH OVERLAP
# ============================================================

def chunk_text(text, chunk_size=800, overlap=200):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap  # move forward with overlap
    return chunks

In [ ]:
# ============================================================
# 🧠 CREATE EMBEDDING
# ============================================================

def get_embedding(text):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding

In [ ]:
# ============================================================
# 🗂️ CHROMADB SETUP
# ============================================================

chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(name="simple_rag")

print("✅ ChromaDB collection ready.")

In [ ]:
# ============================================================
# 📥 STORE CHUNKS
# ============================================================

def add_chunks_to_chroma(chunks):
    for chunk in chunks:
        embedding = get_embedding(chunk)
        collection.add(
            ids=[str(uuid.uuid4())],
            documents=[chunk],
            embeddings=[embedding]
        )
    print(f"✅ Stored {len(chunks)} chunks in ChromaDB.")

In [ ]:
# ============================================================
# 🔎 RETRIEVE DOCUMENTS (returns docs + similarity scores)
# ============================================================

def retrieve(query, top_k=3):
    query_embedding = get_embedding(query)
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=["documents", "distances"]
    )
    docs = results["documents"][0]
    distances = results["distances"][0]
    return docs, distances

In [ ]:
# ============================================================
# 🤖 ASK QUESTION — shows retrieved chunks before answering
# ============================================================

def ask(query, show_chunks=True):
    docs, distances = retrieve(query)

    # ── Display retrieved chunks ────────────────────────────
    if show_chunks:
        print("\n" + "─" * 100)
        print("📄 RETRIEVED CHUNKS")
        print("─" * 100)
        for i, (doc, dist) in enumerate(zip(docs, distances), 1):
            similarity = 1 - dist  # convert distance → similarity score
            print(f"\n[Chunk {i}]  Similarity: {similarity:.4f}")
            print("┄" * 80)
            print(textwrap.fill(doc.strip(), width=96))
        print("\n" + "─" * 100)

    # ── Build prompt & call LLM ─────────────────────────────
    context = "\n\n".join(docs)

    prompt = f"""You are a helpful RAG assistant.

Answer ONLY from the provided context.

If the answer is not in the context, say:
\"I could not find the answer in the document.\"

Context:
{context}

Question:
{query}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You answer questions using retrieved documents."},
            {"role": "user",   "content": prompt}
        ],
        temperature=0.2
    )

    return response.choices[0].message.content

In [ ]:
# ============================================================
# 🏗️ BUILD RAG PIPELINE
# Set PDF_PATH to the path of your PDF file.
# ============================================================

PDF_PATH = "your_document.pdf"   # ← change this

print(f"📂 Loading PDF: {PDF_PATH}")
text = load_pdf(PDF_PATH)
print("✅ PDF loaded.")

print("\n✂️  Chunking document...")
chunks = chunk_text(text, chunk_size=800, overlap=200)
print(f"✅ Created {len(chunks)} chunks.")

print("\n🧠 Generating embeddings and storing in ChromaDB...")
add_chunks_to_chroma(chunks)

print("\n✅ RAG Pipeline Ready")

In [ ]:
# ============================================================
# 💬 INTERACTIVE CHATBOT
# Commands:
#   chunks on   — show retrieved chunks (default)
#   chunks off  — hide retrieved chunks
#   exit        — stop the chatbot
# ============================================================

print("🤖 Chatbot Ready")
print("   Type 'exit' to stop.")
print("   Type 'chunks off' to hide retrieved chunks.")
print("   Type 'chunks on'  to show retrieved chunks.\n")

show_chunks = True

while True:
    try:
        query = input("You: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\nBot: Goodbye.")
        break

    if not query:
        continue

    if query.lower() == "exit":
        print("Bot: Goodbye.")
        break

    if query.lower() == "chunks off":
        show_chunks = False
        print("Bot: Retrieved chunks will now be hidden.\n")
        continue

    if query.lower() == "chunks on":
        show_chunks = True
        print("Bot: Retrieved chunks will now be shown.\n")
        continue

    answer = ask(query, show_chunks=show_chunks)

    print("\n🤖 Answer:")
    print(textwrap.fill(answer, width=100))
    print("\n" + "=" * 100 + "\n")